In [ ]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
print('pandas:', pd.__version__, ' polars:', pl.__version__)

In [ ]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- ultralytics_trainer_read_csv ---
ULTRALYTICS_RESULTS_CSV = "epoch,train_loss,val_loss\n1,0.5,0.6\n2,0.4,0.5\n3,0.35,0.45\n"

self = SimpleNamespace(
    csv=Path("/tmp/results.csv")
)
self.csv.write_text(ULTRALYTICS_RESULTS_CSV, encoding="utf-8")

print("✅ Fixtures loaded")


In [ ]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_ultralytics_trainer_read_csv():
    return pd.read_csv(self.csv).to_dict(orient="list")
    return None

In [ ]:
# ── Generated wrappers (verbatim LLM-generated Polars) ──────────────────────

def gen_ultralytics_trainer_read_csv():

    return pl.read_csv(self.csv).to_dict(as_series=False)

In [ ]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [ ]:
# === Tests: ultralytics_trainer_read_csv ===

def _dict_to_lists(obj):
    if not isinstance(obj, dict):
        return obj
    out = {}
    for key, value in obj.items():
        if isinstance(value, pl.Series):
            out[key] = value.to_list()
        elif isinstance(value, pd.Series):
            out[key] = value.tolist()
        elif isinstance(value, np.ndarray):
            out[key] = value.tolist()
        else:
            out[key] = value
    return out

try:
    _r = gen_ultralytics_trainer_read_csv()
    print("✅ L1 smoke gen_ultralytics_trainer_read_csv: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_ultralytics_trainer_read_csv: {type(_e).__name__}: {_e}")

try:
    _rb = before_ultralytics_trainer_read_csv()
    print("✅ L1 smoke before_ultralytics_trainer_read_csv: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_ultralytics_trainer_read_csv: {type(_e).__name__}: {_e}")

try:
    _rb = _dict_to_lists(before_ultralytics_trainer_read_csv())
    _rg = _dict_to_lists(gen_ultralytics_trainer_read_csv())
    if _rb == _rg:
        print("✅ L2 equivalence ultralytics_trainer_read_csv: MATCH")
    else:
        print(f"❌ L2 equivalence ultralytics_trainer_read_csv: MISMATCH — before={_rb!r}, gen={_rg!r}")
except Exception as _e:
    print(f"❌ L2 equivalence ultralytics_trainer_read_csv: setup error — {type(_e).__name__}: {_e}")

try:
    import tempfile
    _old_csv = self.csv
    with tempfile.TemporaryDirectory() as _tmp:
        self.csv = Path(_tmp) / "results_empty.csv"
        self.csv.write_text("epoch,train_loss,val_loss" + chr(10), encoding="utf-8")
        _rb = _dict_to_lists(before_ultralytics_trainer_read_csv())
        _rg = _dict_to_lists(gen_ultralytics_trainer_read_csv())
    if _rb == _rg == {"epoch": [], "train_loss": [], "val_loss": []}:
        print("✅ L3 edge ultralytics_trainer_read_csv empty CSV: MATCH")
    else:
        print(f"❌ L3 edge ultralytics_trainer_read_csv empty CSV: MISMATCH — before={_rb!r}, gen={_rg!r}")
except Exception as _e:
    print(f"❌ L3 edge ultralytics_trainer_read_csv empty CSV: {type(_e).__name__}: {_e}")
finally:
    self.csv = _old_csv
